In [13]:
# Built-in modules (standard Python libraries)

import os  
# Used to interact with the operating system (files, directories, paths, etc.)

import re  
# Used for working with regular expressions (text cleaning, pattern matching)

import string  
# Provides common string constants (like punctuation, letters, digits)

import warnings  
# Used to control warning messages in Python

from collections import Counter  
# Counter helps count occurrences of elements (e.g., word frequency)

from itertools import combinations  
# Used to generate combinations of elements (e.g., pairs of words)


# External libraries 

import numpy as np  
# Numerical computing library (arrays, math operations)

import pandas as pd  
# Data manipulation library (dataframes, tables, CSV handling)

import matplotlib.pyplot as plt  
# Used for plotting graphs (line plots, histograms, etc.)

import matplotlib.gridspec as gridspec  
# Helps organize multiple plots in a grid layout

import seaborn as sns  
# Advanced visualization library built on matplotlib (better styling, statistical plots)

from wordcloud import WordCloud  
# Used to generate word clouds from text data

from fpdf import FPDF  
# Used to generate PDF files programmatically


# Suppress warning messages
warnings.filterwarnings("ignore")  
# This hides warnings to keep output clean (useful but be careful not to miss important warnings)

In [14]:
# 0. CONFIGURATION CLASS
# This class stores all the parameters/settings for your project in one place
# 👉 This makes your code clean and easy to modify later

class Config:

    CSV_PATH = r"french_data.csv"
    # Path to your dataset file (CSV format)
    # r"" means raw string → useful for file paths (avoids issues with \)

    OUTPUT_DIR = r"output"
    # Folder where all outputs will be saved (plots, PDFs, etc.)

    LANGUAGE_COL = "language"
    # Name of the column that contains the language information

    LANGUAGE_VALUE = "French"
    # The specific language you want to filter (only keep French rows)

    TEXT_COL = "text"
    # Column that contains the text data (important for NLP tasks)

    LABEL_COL = "mental_state"
    # Column that contains labels (e.g., Anxiety, Depression, etc.)

    TOP_N_WORDS = 20
    # Number of most frequent words to display/analyze

    TOP_N_COOC = 20
    # Number of top word co-occurrences (word pairs) to display

1. DATA LOADER

In [15]:


class DataLoader:
    """Loads the CSV and filters to French rows only."""
    # This class is responsible for:
    # 1. Reading the dataset
    # 2. Keeping only rows where language = French

    def __init__(self, cfg: Config):
        self.cfg = cfg
        # Store the configuration object so we can access paths and column names

    def load(self) -> pd.DataFrame:
        # This function loads and processes the dataset

        df = pd.read_csv(self.cfg.CSV_PATH)
        # Read the CSV file using pandas

        print(f"[DataLoader] Total rows loaded       : {len(df)}")
        # Display total number of rows before filtering

        mask = df[self.cfg.LANGUAGE_COL].str.strip().str.lower() == \
               self.cfg.LANGUAGE_VALUE.lower()
        # Create a filter (mask) to keep only rows where:
        # - language column matches "French"
        # - .str.strip() removes extra spaces
        # - .str.lower() ensures case-insensitive comparison

        df = df[mask].copy().reset_index(drop=True)
        # Apply the filter:
        # - df[mask] keeps only matching rows
        # - .copy() avoids potential warnings (good practice)
        # - .reset_index(drop=True) resets row numbers cleanly

        print(f"[DataLoader] French rows kept         : {len(df)}")
        # Display number of rows after filtering

        return df
        # Return the cleaned DataFrame

In [16]:
import re
import string
import pandas as pd

class TextCleaner:
    """
    Cleans the text column and adds helper columns:
      - text_clean      : lowercased, no URLs / punctuation / emojis / emoticons
      - text_nostop     : text_clean without stopwords
      - word_count      : words in cleaned text
      - char_count      : characters in original text
      - punct_count     : punctuation marks in original
      - emoji_count     : emoji characters in original
      - emoticon_count  : text emoticons in original
    """

    EMOJI_RE = re.compile(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF\U000024C2-\U0001F251]+",
        flags=re.UNICODE,
    )

    EMOTICON_RE = re.compile(
        r"(:-\)|:\)|:-\(|:\(|;-\)|;\)|:D|XD|<3|:P|:-P|:O|B\))"
    )

    URL_RE = re.compile(r"https?://\S+|www\.\S+")

    PUNCT_CHARS = set(string.punctuation + "«»–—…")

    # ── constructor ───────────────────────────────────────
    def __init__(self, stopwords_file: str):
        """
        Load stopwords from file + add custom mental-health stopwords
        """

        # Load Kaggle stopwords
        with open(stopwords_file, "r", encoding="utf-8") as f:
            file_stopwords = set(line.strip().lower() for line in f if line.strip())

        # Add YOUR project-specific stopwords
        custom_stopwords = {
            # greetings
            "bonjour", "salut", "merci", "svp", "sil", "plait",

            # consultation words
            "question", "reponse", "docteur", "medecin", "aide", "conseil",

            # vague words
            "chose", "probleme", "situation", "sentiment"
        }

        # FINAL stopwords
        self.FRENCH_STOPWORDS = file_stopwords | custom_stopwords

    # ── counters ──────────────────────────────────────────
    def _count_emojis(self, text: str) -> int:
        return len(self.EMOJI_RE.findall(str(text)))

    def _count_emoticons(self, text: str) -> int:
        return len(self.EMOTICON_RE.findall(str(text)))

    def _count_punct(self, text: str) -> int:
        return sum(1 for c in str(text) if c in self.PUNCT_CHARS)

    # ── main clean ────────────────────────────────────────
    def _clean(self, text: str) -> str:
        text = str(text)
        text = self.URL_RE.sub("", text)
        text = self.EMOJI_RE.sub("", text)
        text = self.EMOTICON_RE.sub("", text)
        text = text.translate(str.maketrans("", "", string.punctuation + "«»–—…"))
        text = re.sub(r"\s+", " ", text).strip().lower()
        return text

    def _remove_stopwords(self, text: str) -> str:
        return " ".join(
            w for w in text.split()
            if w not in self.FRENCH_STOPWORDS and len(w) > 2
        )

    # ── public ────────────────────────────────────────────
    def fit_transform(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        df = df.copy()
        raw = df[text_col].astype(str)

        df["emoji_count"] = raw.apply(self._count_emojis)
        df["emoticon_count"] = raw.apply(self._count_emoticons)
        df["punct_count"] = raw.apply(self._count_punct)
        df["char_count"] = raw.apply(len)

        df["text_clean"] = raw.apply(self._clean)
        df["text_nostop"] = df["text_clean"].apply(self._remove_stopwords)
        df["word_count"] = df["text_clean"].apply(lambda x: len(x.split()))

        before = len(df)
        df.drop_duplicates(subset="text_clean", inplace=True)
        df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)

        print(f"[TextCleaner] Removed {before - len(df)} duplicate/empty rows → {len(df)} remain")
        return df

STOPWORDS_FILE = r"C:\Users\Admin\Documents\FYP\french dataset\french_stpwords.txt"
cleaner = TextCleaner(stopwords_file=STOPWORDS_FILE)
df_clean = cleaner.fit_transform(mydataset, "Question")

In [17]:
class PlotHelper:
    """
    Utility class that centralizes all plotting-related tasks:
    - Applies a consistent visual style to all plots
    - Saves figures to disk
    - Provides helper functions (e.g., safe filenames)
    """

    def __init__(self, cfg: Config):
        """
        Constructor:
        - Receives a configuration object (cfg)
        - Applies global matplotlib styling using values from cfg
        """
        self.cfg = cfg  # store config for later use (paths, colors, DPI, etc.)

        # Update matplotlib global parameters (affects ALL plots)
        plt.rcParams.update({
            "figure.facecolor": cfg.BG,        # background color of the whole figure
            "axes.facecolor":   cfg.BG,        # background color inside the plot area
            "axes.spines.top":  False,         # remove top border of plots
            "axes.spines.right":False,         # remove right border (cleaner look)
            "font.size":        11,            # default font size
        })

    def save(self, filename: str) -> str:
        """
        Saves the current matplotlib figure to the output directory.

        Parameters:
        - filename: name of the file (e.g., "plot.png")

        Returns:
        - full path where the file was saved
        """
        # Build full path using the configured output directory
        path = os.path.join(self.cfg.OUTPUT_DIR, filename)

        # Adjust layout to avoid overlapping labels/titles
        plt.tight_layout()

        # Save figure with high quality (DPI from config)
        plt.savefig(path, dpi=self.cfg.DPI, bbox_inches="tight")

        # Close figure to free memory (important when generating many plots)
        plt.close()

        # Print confirmation
        print(f"  [SAVED] {filename}")

        return path  # return path for reuse if needed

    @staticmethod
    def safe_name(text: str) -> str:
        """
        Converts any string into a safe filename by removing invalid characters.

        Example:
        'Anxiety/Depression?' → 'Anxiety_Depression_'

        This prevents errors when saving files.
        """
        # Replace forbidden characters with "_"
        return re.sub(r'[\\/*?:"<>|]+', "_", str(text)).strip()

In [18]:
class DatasetSize:
    """Req 0 — Size of the dataset."""

    def __init__(self, cfg, helper):
        # cfg → configuration (column names, paths, etc.)
        # helper → PlotHelper (for saving plots if needed)
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> dict:
        """
        Computes general statistics about the dataset
        and returns them as a dictionary.
        """

        # Create a dictionary of dataset metrics
        stats = {
            "Total rows (French)"  : len(df),  # number of rows
            "Unique texts"         : df["text_clean"].nunique(),  # unique cleaned texts
            "Columns"              : len(df.columns),  # number of columns
            "Avg word count"       : round(df["word_count"].mean(), 2),  # average words per text
            "Max word count"       : df["word_count"].max(),  # longest text
            "Min word count"       : df["word_count"].min(),  # shortest text
            "Avg char count"       : round(df["char_count"].mean(), 2),  # avg characters
            "Missing values total" : int(df.isnull().sum().sum()),  # total missing values
        }

        # Print results nicely in console
        print("\n[DatasetSize]")
        for k, v in stats.items():
            print(f"  {k:<28}: {v}")

        return stats  # return results for later use (report, file, etc.)

In [19]:
class LabelDistribution:
    """Req 1 — Distribution of labels."""

    def __init__(self, cfg, helper):
        self.cfg    = cfg   # contains LABEL_COL and color palette
        self.helper = helper  # used to save the figure

    def analyse(self, df: pd.DataFrame) -> str:
        """
        Creates visualizations for label distribution:
        - Bar chart (counts)
        - Pie chart (percentages)
        """
        counts = df[self.cfg.LABEL_COL].value_counts()

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Label Distribution — mental_state", fontsize=14, fontweight="bold")

        # Bar plot
        sns.barplot(x=counts.values, y=counts.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[0])
        axes[0].set_xlabel("Count")
        axes[0].set_title("Count per Label")

        # Pie chart
        axes[1].pie(counts.values, labels=counts.index,
                    autopct="%1.1f%%",
                    colors=sns.color_palette(self.cfg.PALETTE, len(counts)),
                    startangle=140)
        axes[1].set_title("Proportion per Label")

        # Save only **this graph**
        return self.helper.save("01_label_distribution.png")

In [20]:
class TextLengthAnalysis:
    """Req 2 — Text length (words + chars)."""

    def __init__(self, cfg, helper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Text Length Analysis", fontsize=14, fontweight="bold")

        # Histogram of word count
        axes[0].hist(df["word_count"], bins=30, color="#4C9BE8", edgecolor="white")
        axes[0].axvline(df["word_count"].mean(), color="red", linestyle="--",
                        label=f"Mean={df['word_count'].mean():.1f}")
        axes[0].set_title("Word Count Distribution")
        axes[0].set_xlabel("Words per text")
        axes[0].legend()

        # Boxplot of char count by label
        order = (df.groupby(self.cfg.LABEL_COL)["char_count"]
                   .median().sort_values(ascending=False).index)
        sns.boxplot(data=df, x=self.cfg.LABEL_COL, y="char_count",
                    order=order, palette=self.cfg.PALETTE, ax=axes[1])
        axes[1].set_title("Char Count by Label")
        axes[1].set_xlabel("")
        plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

        # Save only **this graph**
        return self.helper.save("02_text_length.png")

In [21]:
class PunctuationAnalysis:
    """Req 3 — Usage of punctuation."""
    # This class focuses on analyzing punctuation usage in your texts.
    # It produces:
    # 1. Histogram of punctuation counts per text
    # 2. Average punctuation per label (bar chart)

    def __init__(self, cfg, helper):
        # cfg → contains LABEL_COL and PALETTE for colors
        # helper → PlotHelper instance used to save figures
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        # Create a figure with 2 subplots side-by-side
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Punctuation Usage", fontsize=14, fontweight="bold")

        # ── Histogram: punctuation per text ─────────────────────────────
        axes[0].hist(df["punct_count"], bins=25, color="#E87B4C", edgecolor="white")
        axes[0].axvline(df["punct_count"].mean(), color="red", linestyle="--",
                        label=f"Mean={df['punct_count'].mean():.1f}")  # mark mean
        axes[0].set_title("Punct Count Distribution")
        axes[0].set_xlabel("Punctuation marks per text")
        axes[0].legend()

        # ── Bar chart: average punctuation by label ─────────────────────
        avg = (df.groupby(self.cfg.LABEL_COL)["punct_count"]
                  .mean().sort_values(ascending=False))  # mean per label
        sns.barplot(x=avg.values, y=avg.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1])
        axes[1].set_title("Avg Punct Count by Label")
        axes[1].set_xlabel("Avg count")

        # Save figure and return the file path
        return self.helper.save("03_punctuation.png")

In [22]:
class WordCloudAnalysis:
    """Req 4 — Word Clouds per label."""
    # This class generates word clouds for each label in your dataset.
    # Each word cloud is based on text after stopwords removal (text_nostop).

    def __init__(self, cfg, helper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> list[str]:
        # Get unique labels
        labels  = df[self.cfg.LABEL_COL].unique()
        n       = len(labels)
        cols    = min(3, n)                 # max 3 columns
        rows    = (n + cols - 1) // cols    # compute number of rows
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
        axes    = np.array(axes).flatten()  # flatten in case of multiple rows
        fig.suptitle("Word Clouds per Label", fontsize=15, fontweight="bold")

        # Define color maps for word clouds
        cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "YlOrBr"]

        for i, label in enumerate(labels):
            # Concatenate all text for the current label
            text = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"])
            if not text.strip():               # skip empty text
                axes[i].axis("off")
                continue

            # Generate word cloud
            wc = WordCloud(
                width=600, height=350,
                background_color="white",
                colormap=cmaps[i % len(cmaps)],
                max_words=100,
                collocations=False,            # no repeated phrases
            ).generate(text)

            # Display word cloud
            axes[i].imshow(wc, interpolation="bilinear")
            axes[i].axis("off")
            axes[i].set_title(str(label), fontsize=12, fontweight="bold")

        # Turn off any unused axes
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        # Save figure and return list of paths
        path = self.helper.save("04_wordclouds_per_label.png")
        return [path]

In [23]:
class CoOccurrenceAnalysis:
    """Req 5 — Co-occurrence of words per label."""
    # This class computes which word pairs appear together most frequently for each label.
    # Graph: Horizontal Bar Chart showing top N co-occurring word pairs per label.

    def __init__(self, cfg, helper):
        self.cfg    = cfg      # Config containing LABEL_COL and TOP_N_COOC
        self.helper = helper   # PlotHelper instance to save figures

    def _cooccurrence(self, texts, top_n):
        """
        Compute co-occurrence counts for word pairs in a list of texts.
        - Each text: words are unique within the sentence (set)
        - Returns the top_n most common pairs
        """
        co = Counter()
        for sentence in texts:
            words = list(set(sentence.split()))   # unique words in the sentence
            for pair in combinations(sorted(words), 2):
                co[pair] += 1
        return co.most_common(top_n)

    def analyse(self, df: pd.DataFrame) -> list[str]:
        paths  = []
        labels = df[self.cfg.LABEL_COL].unique()

        for label in labels:
            texts = df[df[self.cfg.LABEL_COL] == label]["text_nostop"]
            pairs = self._cooccurrence(texts, self.cfg.TOP_N_COOC)
            if not pairs:
                continue

            # Prepare labels and counts for plotting
            pair_labels = [f"{a} & {b}" for (a, b), _ in pairs]
            counts      = [c for _, c in pairs]

            # Plot horizontal bar chart
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(x=counts, y=pair_labels, palette="mako", ax=ax)
            ax.set_title(f"Top Word Co-occurrences — {label}",
                         fontsize=13, fontweight="bold")
            ax.set_xlabel("Co-occurrence count")

            # Save figure
            fname = f"05_cooccurrence_{self.helper.safe_name(label)}.png"
            paths.append(self.helper.save(fname))

        return paths

In [24]:
class CommonWordsAnalysis:
    """Req 6 — Common words per label."""
    # This class shows the top N most frequent words per label.
    # Graph: Horizontal Bar Chart per label.

    def __init__(self, cfg, helper):
        self.cfg    = cfg      # Config containing LABEL_COL and TOP_N_WORDS
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> list[str]:
        paths  = []
        labels = df[self.cfg.LABEL_COL].unique()
        cols   = min(2, len(labels))                        # max 2 columns
        rows   = (len(labels) + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 5))
        axes = np.array(axes).flatten()
        fig.suptitle(f"Top {self.cfg.TOP_N_WORDS} Common Words per Label",
                     fontsize=14, fontweight="bold")

        for i, label in enumerate(labels):
            # Concatenate all texts, split into words, count frequency
            words = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"]).split()
            freq  = Counter(words).most_common(self.cfg.TOP_N_WORDS)
            if not freq:
                axes[i].axis("off")
                continue
            w, c = zip(*freq)

            # Horizontal bar chart
            sns.barplot(x=list(c), y=list(w), palette="rocket", ax=axes[i])
            axes[i].set_title(str(label), fontsize=11, fontweight="bold")
            axes[i].set_xlabel("Frequency")

        # Turn off unused axes
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        # Save figure
        path = self.helper.save("06_common_words_per_label.png")
        return [path]

In [25]:
class CommonWordsAnalysis:
    """Req 6 — Common words per label."""
    # This class shows the top N most frequent words per label.
    # Graph: Horizontal Bar Chart per label.

    def __init__(self, cfg, helper):
        self.cfg    = cfg      # Config containing LABEL_COL and TOP_N_WORDS
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> list[str]:
        paths  = []
        labels = df[self.cfg.LABEL_COL].unique()
        cols   = min(2, len(labels))                        # max 2 columns
        rows   = (len(labels) + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 5))
        axes = np.array(axes).flatten()
        fig.suptitle(f"Top {self.cfg.TOP_N_WORDS} Common Words per Label",
                     fontsize=14, fontweight="bold")

        for i, label in enumerate(labels):
            # Concatenate all texts, split into words, count frequency
            words = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"]).split()
            freq  = Counter(words).most_common(self.cfg.TOP_N_WORDS)
            if not freq:
                axes[i].axis("off")
                continue
            w, c = zip(*freq)

            # Horizontal bar chart
            sns.barplot(x=list(c), y=list(w), palette="rocket", ax=axes[i])
            axes[i].set_title(str(label), fontsize=11, fontweight="bold")
            axes[i].set_xlabel("Frequency")

        # Turn off unused axes
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        # Save figure
        path = self.helper.save("06_common_words_per_label.png")
        return [path]

In [26]:
class EDAOrchestrator:
    """
    Wires everything together.
    Call .run() to execute the full pipeline.
    """

    def __init__(self, cfg: Config):
        self.cfg     = cfg
        self.helper  = PlotHelper(cfg)
        os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    def run(self):
        print("=" * 55)
        print("  FRENCH EDA PIPELINE — START")
        print("=" * 55)

        # ── load ───────────────────────────────────────────
        df = DataLoader(self.cfg).load()

        # ── clean ──────────────────────────────────────────
        df = TextCleaner().fit_transform(df, self.cfg.TEXT_COL)

        # ── save cleaned csv ───────────────────────────────
        clean_path = os.path.join(self.cfg.OUTPUT_DIR, "french_cleaned.csv")
        df.to_csv(clean_path, index=False, encoding="utf-8-sig")
        print(f"[Pipeline] Cleaned CSV saved → {clean_path}")

        # ── analyses ───────────────────────────────────────
        print("\n[Pipeline] Running EDA analyses...")

        stats = DatasetSize(self.cfg, self.helper).analyse(df)

        plot_map = [
            ("1. Label Distribution",
             LabelDistribution(self.cfg, self.helper).analyse(df)),

            ("2. Text Length Analysis",
             TextLengthAnalysis(self.cfg, self.helper).analyse(df)),

            ("3. Punctuation Usage",
             PunctuationAnalysis(self.cfg, self.helper).analyse(df)),

            ("4. Word Clouds per Label",
             WordCloudAnalysis(self.cfg, self.helper).analyse(df)),

            ("5. Word Co-occurrence per Label",
             CoOccurrenceAnalysis(self.cfg, self.helper).analyse(df)),

            ("6. Common Words per Label",
             CommonWordsAnalysis(self.cfg, self.helper).analyse(df)),

            ("7 & 8. Emoji & Emoticon Analysis",
             EmojiEmoticonAnalysis(self.cfg, self.helper).analyse(df)),
        ]